In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from sqlalchemy import create_engine, URL, text

In [3]:
load_dotenv(Path("../.env"))

print(os.getenv("DB_HOST"))
print(os.getenv("DB_NAME"))
print(os.getenv("DB_USER"))

localhost
ecommerce_analytics
ecommerce_user


In [4]:
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(db_url)

In [5]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_user, current_database();")
    )

    print(result.fetchone())

('ecommerce_user', 'ecommerce_analytics')


In [6]:
import pandas as pd

customers = pd.read_csv(
    "../data/raw/olist_customers_dataset.csv"
)

print(customers.shape)
customers.head()

(99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [7]:
# into PostgreSQL

customers.to_sql(name="customers",con=engine,schema="raw",if_exists="replace",index=False)

441

In [8]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM raw.customers;"))
    print(result.scalar())

99441


In [9]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

print(orders.shape)
orders.head()

(99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [10]:
#into postgre

orders.to_sql(name="orders",con=engine,schema="raw",if_exists="replace",index=False)

441

In [11]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM raw.orders;"))

    print(result.scalar())

99441


In [12]:
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

print(order_items.shape)
order_items.head()

(112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [13]:
order_items.to_sql(name="order_items",con=engine,schema="raw",if_exists="replace",index=False)

650

In [14]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM raw.order_items;"))

    print(result.scalar())

112650


In [15]:
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

print(products.shape)
products.head()

(32951, 9)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [16]:
products.to_sql(name="products",con=engine,schema="raw",if_exists="replace",index=False)

951

In [17]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM raw.products;"))

    print(result.scalar())

32951


In [18]:
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

print(payments.shape)
payments.head()

(103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [19]:
payments.to_sql(name="payments",con=engine,schema="raw",if_exists="replace",index=False)

886

In [20]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM raw.payments;"))

    print(result.scalar())

103886


In [21]:
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

print(reviews.shape)
reviews.head()

(99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [22]:
reviews.to_sql(name="reviews",con=engine,schema="raw",if_exists="replace",index=False)

224

In [23]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM raw.reviews;"))

    print(result.scalar())

99224


In [24]:
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

print(sellers.shape)
sellers.head()

(3095, 4)


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [25]:
sellers.to_sql(name="sellers",con=engine,schema="raw",if_exists="replace",index=False)

95

In [26]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM raw.sellers;"))

    print(result.scalar())

3095


In [27]:
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

print(geolocation.shape)
geolocation.head()

(1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [28]:
geolocation.to_sql(name="geolocation",con=engine,schema="raw",if_exists="replace",index=False,chunksize=10000)

100163

In [29]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM raw.geolocation;"))

    print(result.scalar())

1000163


In [30]:
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

print(category_translation.shape)
category_translation.head()

(71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [31]:
category_translation.to_sql(name="category_translation",con=engine,schema="raw",if_exists="replace",index=False)

71

In [32]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM raw.category_translation;"))

    print(result.scalar())

71


In [33]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            table_name
        FROM information_schema.tables
        WHERE table_schema = 'raw'
        ORDER BY table_name;
    """))

    for row in result:
        print(row[0])

category_translation
customers
geolocation
order_items
orders
payments
products
reviews
sellers


In [34]:
#all row counts together

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM raw.customers
        UNION ALL
        SELECT 'orders', COUNT(*) FROM raw.orders
        UNION ALL
        SELECT 'order_items', COUNT(*) FROM raw.order_items
        UNION ALL
        SELECT 'products', COUNT(*) FROM raw.products
        UNION ALL
        SELECT 'payments', COUNT(*) FROM raw.payments
        UNION ALL
        SELECT 'reviews', COUNT(*) FROM raw.reviews
        UNION ALL
        SELECT 'sellers', COUNT(*) FROM raw.sellers
        UNION ALL
        SELECT 'geolocation', COUNT(*) FROM raw.geolocation
        UNION ALL
        SELECT 'category_translation', COUNT(*) FROM raw.category_translation;
    """))

    for row in result:
        print(row)

('category_translation', 71)
('sellers', 3095)
('products', 32951)
('payments', 103886)
('customers', 99441)
('reviews', 99224)
('geolocation', 1000163)
('order_items', 112650)
('orders', 99441)


In [35]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw'
          AND table_name = 'orders'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('order_id', 'text')
('customer_id', 'text')
('order_status', 'text')
('order_purchase_timestamp', 'text')
('order_approved_at', 'text')
('order_delivered_carrier_date', 'text')
('order_delivered_customer_date', 'text')
('order_estimated_delivery_date', 'text')


In [36]:
#Creating a staging schema

with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS staging;"))

print("staging schema created")

staging schema created


In [37]:
with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS staging.orders;"))

    connection.execute(text("""
        CREATE TABLE staging.orders AS
        SELECT
            order_id,
            customer_id,
            order_status,
            order_purchase_timestamp::timestamp
                AS order_purchase_timestamp,
            order_approved_at::timestamp
                AS order_approved_at,
            order_delivered_carrier_date::timestamp
                AS order_delivered_carrier_date,
            order_delivered_customer_date::timestamp
                AS order_delivered_customer_date,
            order_estimated_delivery_date::timestamp
                AS order_estimated_delivery_date
        FROM raw.orders;
    """))

print("staging.orders created")

staging.orders created


In [38]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'staging'
          AND table_name = 'orders'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('order_id', 'text')
('customer_id', 'text')
('order_status', 'text')
('order_purchase_timestamp', 'timestamp without time zone')
('order_approved_at', 'timestamp without time zone')
('order_delivered_carrier_date', 'timestamp without time zone')
('order_delivered_customer_date', 'timestamp without time zone')
('order_estimated_delivery_date', 'timestamp without time zone')


In [39]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM raw.orders) AS raw_count,
            (SELECT COUNT(*) FROM staging.orders) AS staging_count;
    """))

    print(result.fetchone())

(99441, 99441)


In [40]:
#Inspect raw.customers data types

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw'
          AND table_name = 'customers'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('customer_id', 'text')
('customer_unique_id', 'text')
('customer_zip_code_prefix', 'bigint')
('customer_city', 'text')
('customer_state', 'text')


In [41]:
with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS staging.customers;")
    )

    connection.execute(text("""
        CREATE TABLE staging.customers AS
        SELECT
            customer_id,
            customer_unique_id,
            LPAD(customer_zip_code_prefix::text, 5, '0')
                AS customer_zip_code_prefix,
            TRIM(customer_city) AS customer_city,
            TRIM(customer_state) AS customer_state
        FROM raw.customers;
    """))

print("staging.customers created")

staging.customers created


In [42]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'staging'
          AND table_name = 'customers'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('customer_id', 'text')
('customer_unique_id', 'text')
('customer_zip_code_prefix', 'text')
('customer_city', 'text')
('customer_state', 'text')


In [43]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM raw.customers) AS raw_count,
            (SELECT COUNT(*) FROM staging.customers) AS staging_count;
    """))

    print(result.fetchone())

(99441, 99441)


In [44]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw'
          AND table_name = 'order_items'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('order_id', 'text')
('order_item_id', 'bigint')
('product_id', 'text')
('seller_id', 'text')
('shipping_limit_date', 'text')
('price', 'double precision')
('freight_value', 'double precision')


In [45]:
with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS staging.order_items;")
    )

    connection.execute(text("""
        CREATE TABLE staging.order_items AS
        SELECT
            order_id,
            order_item_id,
            product_id,
            seller_id,
            shipping_limit_date::timestamp
                AS shipping_limit_date,
            price::numeric(12, 2)
                AS price,
            freight_value::numeric(12, 2)
                AS freight_value
        FROM raw.order_items;
    """))

print("staging.order_items created")

staging.order_items created


In [46]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type,
            numeric_precision,
            numeric_scale
        FROM information_schema.columns
        WHERE table_schema = 'staging'
          AND table_name = 'order_items'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('order_id', 'text', None, None)
('order_item_id', 'bigint', 64, 0)
('product_id', 'text', None, None)
('seller_id', 'text', None, None)
('shipping_limit_date', 'timestamp without time zone', None, None)
('price', 'numeric', 12, 2)
('freight_value', 'numeric', 12, 2)


In [47]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM raw.order_items) AS raw_count,
            (SELECT COUNT(*) FROM staging.order_items) AS staging_count;
    """))

    print(result.fetchone())

(112650, 112650)


In [48]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw'
          AND table_name = 'products'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('product_id', 'text')
('product_category_name', 'text')
('product_name_lenght', 'double precision')
('product_description_lenght', 'double precision')
('product_photos_qty', 'double precision')
('product_weight_g', 'double precision')
('product_length_cm', 'double precision')
('product_height_cm', 'double precision')
('product_width_cm', 'double precision')


In [49]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) FILTER (
                WHERE product_name_lenght IS NOT NULL
                  AND product_name_lenght <> FLOOR(product_name_lenght)
            ) AS name_length_decimals,

            COUNT(*) FILTER (
                WHERE product_description_lenght IS NOT NULL
                  AND product_description_lenght <> FLOOR(product_description_lenght)
            ) AS description_length_decimals,

            COUNT(*) FILTER (
                WHERE product_photos_qty IS NOT NULL
                  AND product_photos_qty <> FLOOR(product_photos_qty)
            ) AS photos_qty_decimals,

            COUNT(*) FILTER (
                WHERE product_weight_g IS NOT NULL
                  AND product_weight_g <> FLOOR(product_weight_g)
            ) AS weight_decimals,

            COUNT(*) FILTER (
                WHERE product_length_cm IS NOT NULL
                  AND product_length_cm <> FLOOR(product_length_cm)
            ) AS length_decimals,

            COUNT(*) FILTER (
                WHERE product_height_cm IS NOT NULL
                  AND product_height_cm <> FLOOR(product_height_cm)
            ) AS height_decimals,

            COUNT(*) FILTER (
                WHERE product_width_cm IS NOT NULL
                  AND product_width_cm <> FLOOR(product_width_cm)
            ) AS width_decimals

        FROM raw.products;
    """))

    print(result.fetchone())

(0, 0, 0, 0, 0, 0, 0)


In [50]:
with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS staging.products;")
    )

    connection.execute(text("""
        CREATE TABLE staging.products AS
        SELECT
            product_id,

            NULLIF(TRIM(product_category_name), '')
                AS product_category_name,

            product_name_lenght::integer
                AS product_name_length,

            product_description_lenght::integer
                AS product_description_length,

            product_photos_qty::integer
                AS product_photos_qty,

            product_weight_g::integer
                AS product_weight_g,

            product_length_cm::integer
                AS product_length_cm,

            product_height_cm::integer
                AS product_height_cm,

            product_width_cm::integer
                AS product_width_cm

        FROM raw.products;
    """))

print("staging.products created")

staging.products created


In [51]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'staging'
          AND table_name = 'products'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('product_id', 'text')
('product_category_name', 'text')
('product_name_length', 'integer')
('product_description_length', 'integer')
('product_photos_qty', 'integer')
('product_weight_g', 'integer')
('product_length_cm', 'integer')
('product_height_cm', 'integer')
('product_width_cm', 'integer')


In [52]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM raw.products) AS raw_count,
            (SELECT COUNT(*) FROM staging.products) AS staging_count;
    """))

    print(result.fetchone())

(32951, 32951)


In [53]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw'
          AND table_name = 'payments'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('order_id', 'text')
('payment_sequential', 'bigint')
('payment_type', 'text')
('payment_installments', 'bigint')
('payment_value', 'double precision')


In [54]:
with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS staging.payments;")
    )

    connection.execute(text("""
        CREATE TABLE staging.payments AS
        SELECT
            order_id,

            payment_sequential::integer
                AS payment_sequential,

            NULLIF(TRIM(payment_type), '')
                AS payment_type,

            payment_installments::integer
                AS payment_installments,

            payment_value::numeric(12, 2)
                AS payment_value

        FROM raw.payments;
    """))

print("staging.payments created")

staging.payments created


In [55]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type,
            numeric_precision,
            numeric_scale
        FROM information_schema.columns
        WHERE table_schema = 'staging'
          AND table_name = 'payments'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('order_id', 'text', None, None)
('payment_sequential', 'integer', 32, 0)
('payment_type', 'text', None, None)
('payment_installments', 'integer', 32, 0)
('payment_value', 'numeric', 12, 2)


In [56]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM raw.payments) AS raw_count,
            (SELECT COUNT(*) FROM staging.payments) AS staging_count;
    """))

    print(result.fetchone())

(103886, 103886)


In [57]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw'
          AND table_name = 'reviews'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)
        

('review_id', 'text')
('order_id', 'text')
('review_score', 'bigint')
('review_comment_title', 'text')
('review_comment_message', 'text')
('review_creation_date', 'text')
('review_answer_timestamp', 'text')


In [58]:
with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS staging.reviews;")
    )

    connection.execute(text("""
        CREATE TABLE staging.reviews AS
        SELECT
            review_id,
            order_id,

            review_score::integer
                AS review_score,

            NULLIF(TRIM(review_comment_title), '')
                AS review_comment_title,

            NULLIF(TRIM(review_comment_message), '')
                AS review_comment_message,

            review_creation_date::date
                AS review_creation_date,

            review_answer_timestamp::timestamp
                AS review_answer_timestamp

        FROM raw.reviews;
    """))

print("staging.reviews created")

staging.reviews created


In [59]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'staging'
          AND table_name = 'reviews'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('review_id', 'text')
('order_id', 'text')
('review_score', 'integer')
('review_comment_title', 'text')
('review_comment_message', 'text')
('review_creation_date', 'date')
('review_answer_timestamp', 'timestamp without time zone')


In [60]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM raw.reviews) AS raw_count,
            (SELECT COUNT(*) FROM staging.reviews) AS staging_count;
    """))

    print(result.fetchone())

(99224, 99224)


In [61]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw'
          AND table_name = 'sellers'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('seller_id', 'text')
('seller_zip_code_prefix', 'bigint')
('seller_city', 'text')
('seller_state', 'text')


In [62]:
with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS staging.sellers;")
    )

    connection.execute(text("""
        CREATE TABLE staging.sellers AS
        SELECT
            seller_id,

            LPAD(seller_zip_code_prefix::text, 5, '0')
                AS seller_zip_code_prefix,

            TRIM(seller_city)
                AS seller_city,

            TRIM(seller_state)
                AS seller_state

        FROM raw.sellers;
    """))

print("staging.sellers created")

staging.sellers created


In [63]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'staging'
          AND table_name = 'sellers'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('seller_id', 'text')
('seller_zip_code_prefix', 'text')
('seller_city', 'text')
('seller_state', 'text')


In [64]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM raw.sellers) AS raw_count,
            (SELECT COUNT(*) FROM staging.sellers) AS staging_count;
    """))

    print(result.fetchone())

(3095, 3095)


In [65]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw'
          AND table_name = 'geolocation'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('geolocation_zip_code_prefix', 'bigint')
('geolocation_lat', 'double precision')
('geolocation_lng', 'double precision')
('geolocation_city', 'text')
('geolocation_state', 'text')


In [66]:
with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS staging.geolocation;")
    )

    connection.execute(text("""
        CREATE TABLE staging.geolocation AS
        SELECT
            LPAD(geolocation_zip_code_prefix::text, 5, '0')
                AS geolocation_zip_code_prefix,

            geolocation_lat,

            geolocation_lng,

            TRIM(geolocation_city)
                AS geolocation_city,

            TRIM(geolocation_state)
                AS geolocation_state

        FROM raw.geolocation;
    """))

print("staging.geolocation created")

staging.geolocation created


In [67]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'staging'
          AND table_name = 'geolocation'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('geolocation_zip_code_prefix', 'text')
('geolocation_lat', 'double precision')
('geolocation_lng', 'double precision')
('geolocation_city', 'text')
('geolocation_state', 'text')


In [68]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM raw.geolocation) AS raw_count,
            (SELECT COUNT(*) FROM staging.geolocation) AS staging_count;
    """))

    print(result.fetchone())

(1000163, 1000163)


In [69]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw'
          AND table_name = 'category_translation'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('product_category_name', 'text')
('product_category_name_english', 'text')


In [70]:
with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS staging.category_translation;")
    )

    connection.execute(text("""
        CREATE TABLE staging.category_translation AS
        SELECT
            NULLIF(TRIM(product_category_name), '')
                AS product_category_name,

            NULLIF(TRIM(product_category_name_english), '')
                AS product_category_name_english

        FROM raw.category_translation;
    """))

print("staging.category_translation created")

staging.category_translation created


In [71]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'staging'
          AND table_name = 'category_translation'
        ORDER BY ordinal_position;
    """))

    for row in result:
        print(row)

('product_category_name', 'text')
('product_category_name_english', 'text')


In [72]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM raw.category_translation) AS raw_count,
            (SELECT COUNT(*) FROM staging.category_translation) AS staging_count;
    """))

    print(result.fetchone())

(71, 71)


In [73]:
#Final staging row-count check

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT 'customers' AS table_name, COUNT(*) FROM staging.customers
        UNION ALL
        SELECT 'orders', COUNT(*) FROM staging.orders
        UNION ALL
        SELECT 'order_items', COUNT(*) FROM staging.order_items
        UNION ALL
        SELECT 'products', COUNT(*) FROM staging.products
        UNION ALL
        SELECT 'payments', COUNT(*) FROM staging.payments
        UNION ALL
        SELECT 'reviews', COUNT(*) FROM staging.reviews
        UNION ALL
        SELECT 'sellers', COUNT(*) FROM staging.sellers
        UNION ALL
        SELECT 'geolocation', COUNT(*) FROM staging.geolocation
        UNION ALL
        SELECT 'category_translation', COUNT(*) FROM staging.category_translation;
    """))

    for row in result:
        print(row)

('category_translation', 71)
('sellers', 3095)
('products', 32951)
('customers', 99441)
('payments', 103886)
('reviews', 99224)
('orders', 99441)
('order_items', 112650)
('geolocation', 1000163)


In [74]:
#Test the orders primary key :  quality test

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT order_id) AS unique_order_ids,
            COUNT(*) FILTER (WHERE order_id IS NULL) AS null_order_ids
        FROM staging.orders;
    """))

    print(result.fetchone())

(99441, 99441, 0)


In [75]:
#testing that every orders.customer_id actually exists in staging.customers.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS unmatched_orders
        FROM staging.orders o
        LEFT JOIN staging.customers c
            ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL;
    """))

    print('Number of orders pointing to a customer that does not exist =',result.scalar())

Number of orders pointing to a customer that does not exist = 0


In [76]:
#checking whether every item belongs to an order that actually exists.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS unmatched_order_items
        FROM staging.order_items oi
        LEFT JOIN staging.orders o
            ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL;
    """))

    print("Number of order-item rows whose order_id does not exist in the orders table = ",result.scalar())

Number of order-item rows whose order_id does not exist in the orders table =  0


In [77]:
#checking that every product_id in staging.order_items exists in staging.products.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS unmatched_products
        FROM staging.order_items oi
        LEFT JOIN staging.products p
            ON oi.product_id = p.product_id
        WHERE p.product_id IS NULL;
    """))

    print("Number of order items referencing a product that does not exist? = ",result.scalar())

Number of order items referencing a product that does not exist? =  0


In [78]:
#testing that every seller referenced in staging.order_items exists in staging.sellers.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS unmatched_sellers
        FROM staging.order_items oi
        LEFT JOIN staging.sellers s
            ON oi.seller_id = s.seller_id
        WHERE s.seller_id IS NULL;
    """))

    print(result.scalar())

0


In [79]:
#testing whether every payment belongs to a valid order.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS unmatched_payments
        FROM staging.payments p
        LEFT JOIN staging.orders o
            ON p.order_id = o.order_id
        WHERE o.order_id IS NULL;
    """))

    print(result.scalar())

0


In [80]:
#testing that every review is linked to an existing order.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS unmatched_reviews
        FROM staging.reviews r
        LEFT JOIN staging.orders o
            ON r.order_id = o.order_id
        WHERE o.order_id IS NULL;
    """))

    print(result.scalar())

0


In [81]:
#testing whether every non-null Portuguese product category has an English translation.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS products_without_translation
        FROM staging.products p
        LEFT JOIN staging.category_translation ct
            ON p.product_category_name = ct.product_category_name
        WHERE p.product_category_name IS NOT NULL
          AND ct.product_category_name IS NULL;
    """))

    print(result.scalar())

13


In [82]:
#Identify the missing category names

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            p.product_category_name,
            COUNT(*) AS product_count
        FROM staging.products p
        LEFT JOIN staging.category_translation ct
            ON p.product_category_name = ct.product_category_name
        WHERE p.product_category_name IS NOT NULL
          AND ct.product_category_name IS NULL
        GROUP BY p.product_category_name
        ORDER BY product_count DESC;
    """))

    for row in result:
        print(row)

('portateis_cozinha_e_preparadores_de_alimentos', 10)
('pc_gamer', 3)


In [83]:
#test whether customer ZIP prefixes can be matched to our geolocation data.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS unmatched_customers
        FROM staging.customers c

        LEFT JOIN (
            SELECT DISTINCT geolocation_zip_code_prefix
            FROM staging.geolocation
        ) g
            ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix

        WHERE g.geolocation_zip_code_prefix IS NULL;
    """))

    print(result.scalar())

278


In [84]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS unmatched_sellers
        FROM staging.sellers s

        LEFT JOIN (
            SELECT DISTINCT geolocation_zip_code_prefix
            FROM staging.geolocation
        ) g
            ON s.seller_zip_code_prefix = g.geolocation_zip_code_prefix

        WHERE g.geolocation_zip_code_prefix IS NULL;
    """))

    print(result.scalar())

#This means only 7 sellers have ZIP prefixes that cannot be matched to the geolocation table.

7


In [85]:
#Test customers.customer_id as a key
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS unique_customer_ids,
            COUNT(*) FILTER (WHERE customer_id IS NULL) AS null_customer_ids
        FROM staging.customers;
    """))

    print(result.fetchone())

(99441, 99441, 0)


In [86]:
#Test products.product_id as a key

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT product_id) AS unique_product_ids,
            COUNT(*) FILTER (WHERE product_id IS NULL) AS null_product_ids
        FROM staging.products;
    """))

    print(result.fetchone())

(32951, 32951, 0)


In [87]:
#Test sellers.seller_id as a key

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT seller_id) AS unique_seller_ids,
            COUNT(*) FILTER (WHERE seller_id IS NULL) AS null_seller_ids
        FROM staging.sellers;
    """))

    print(result.fetchone())

(3095, 3095, 0)


In [88]:
#Test the reviews key structure: (review_id, order_id) together should uniquely identify each row

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT review_id) AS unique_review_ids,
            COUNT(DISTINCT (review_id, order_id)) AS unique_review_order_pairs
        FROM staging.reviews;
    """))

    print(result.fetchone())
    

(99224, 98410, 99224)


In [89]:
#checking (order_id, payment_sequential) as primary key

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT order_id) AS unique_orders,
            COUNT(DISTINCT (order_id, payment_sequential)) AS unique_payment_pairs,
            COUNT(*) FILTER (
                WHERE order_id IS NULL
                   OR payment_sequential IS NULL
            ) AS null_key_rows
        FROM staging.payments;
    """))

    print(result.fetchone())

(103886, 99440, 103886, 0)


In [90]:
#Test the order_items key structure: (order_id, order_item_id) as primary key

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT order_id) AS unique_orders,
            COUNT(DISTINCT (order_id, order_item_id)) AS unique_item_pairs,
            COUNT(*) FILTER (
                WHERE order_id IS NULL
                   OR order_item_id IS NULL
            ) AS null_key_rows
        FROM staging.order_items;
    """))

    print(result.fetchone())

(112650, 98666, 112650, 0)


In [91]:
#Test category_translation key uniqueness

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT product_category_name) AS unique_categories,
            COUNT(*) FILTER (
                WHERE product_category_name IS NULL
            ) AS null_categories
        FROM staging.category_translation;
    """))

    print(result.fetchone())

(71, 71, 0)


In [92]:
#Validate review scores: valid scores should be from 1 to 5.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            MIN(review_score) AS min_score,
            MAX(review_score) AS max_score,
            COUNT(*) FILTER (
                WHERE review_score < 1
                   OR review_score > 5
                   OR review_score IS NULL
            ) AS invalid_scores
        FROM staging.reviews;
    """))

    print(result.fetchone())



(1, 5, 0)


In [93]:
#Validate payment values: whether payment_value contains any negative or null amounts.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            MIN(payment_value) AS min_payment,
            MAX(payment_value) AS max_payment,
            COUNT(*) FILTER (WHERE payment_value < 0 OR payment_value IS NULL) AS invalid_payments
        FROM staging.payments;
    """))

    print(result.fetchone())

(Decimal('0.00'), Decimal('13664.08'), 0)


In [94]:
#Validate payment installments

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            MIN(payment_installments) AS min_installments,
            MAX(payment_installments) AS max_installments,
            COUNT(*) FILTER (
                WHERE payment_installments = 0
            ) AS zero_installment_rows,
            COUNT(*) FILTER (
                WHERE payment_installments < 0
                   OR payment_installments IS NULL
            ) AS invalid_installments
        FROM staging.payments;
    """))

    print(result.fetchone())

(0, 24, 2, 0)


In [95]:
#Validate order-item prices: whether price or freight_value contain negative or null values.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            MIN(price) AS min_price,
            MAX(price) AS max_price,
            COUNT(*) FILTER (
                WHERE price < 0
                   OR price IS NULL
            ) AS invalid_prices,

            MIN(freight_value) AS min_freight,
            MAX(freight_value) AS max_freight,
            COUNT(*) FILTER (
                WHERE freight_value < 0
                   OR freight_value IS NULL
            ) AS invalid_freight

        FROM staging.order_items;
    """))

    print(result.fetchone())



(Decimal('0.85'), Decimal('6735.00'), 0, Decimal('0.00'), Decimal('409.68'), 0)


In [96]:
#Validate order timestamp sequence: whether important order timestamps appear in a sensible order.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) FILTER (
                WHERE order_approved_at IS NOT NULL
                  AND order_approved_at < order_purchase_timestamp
            ) AS approval_before_purchase,

            COUNT(*) FILTER (
                WHERE order_delivered_carrier_date IS NOT NULL
                  AND order_approved_at IS NOT NULL
                  AND order_delivered_carrier_date < order_approved_at
            ) AS carrier_before_approval,

            COUNT(*) FILTER (
                WHERE order_delivered_customer_date IS NOT NULL
                  AND order_delivered_carrier_date IS NOT NULL
                  AND order_delivered_customer_date < order_delivered_carrier_date
            ) AS customer_before_carrier

        FROM staging.orders;
    """))

    print(result.fetchone())

    

(0, 1359, 23)


In [97]:
#Understanding the 1,359 carrier-before-approval cases:
#  Let’s see whether these are mostly just a few hours apart on the same day, or larger inconsistencies.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) FILTER (
                WHERE order_delivered_carrier_date < order_approved_at
                  AND order_delivered_carrier_date::date = order_approved_at::date
            ) AS same_day_anomalies,

            COUNT(*) FILTER (
                WHERE order_delivered_carrier_date < order_approved_at
                  AND order_delivered_carrier_date::date < order_approved_at::date
            ) AS different_day_anomalies,

            MAX(
                order_approved_at - order_delivered_carrier_date
            ) FILTER (
                WHERE order_delivered_carrier_date < order_approved_at
            ) AS maximum_gap

        FROM staging.orders;
    """))

    print(result.fetchone())

(679, 680, datetime.timedelta(days=171, seconds=18922))


In [98]:
#checking the largest anomalies

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            order_id,
            order_status,
            order_purchase_timestamp,
            order_approved_at,
            order_delivered_carrier_date,
            order_delivered_customer_date,
            order_approved_at - order_delivered_carrier_date AS gap
        FROM staging.orders
        WHERE order_delivered_carrier_date < order_approved_at
        ORDER BY gap DESC
        LIMIT 10;
    """))

    for row in result:
        print(row)

('7c48bb55e8e4f7e56d412e9653db37bc', 'delivered', datetime.datetime(2018, 7, 16, 18, 40, 53), datetime.datetime(2018, 7, 16, 18, 50, 22), datetime.datetime(2018, 1, 26, 13, 35), datetime.datetime(2018, 7, 23, 20, 4, 45), datetime.timedelta(days=171, seconds=18922))
('1fab4ac9d85079b3da72a11475ae1685', 'delivered', datetime.datetime(2017, 9, 1, 19, 4, 22), datetime.datetime(2017, 9, 13, 22, 6, 11), datetime.datetime(2017, 9, 4, 13, 10, 23), datetime.datetime(2017, 9, 8, 20, 13, 3), datetime.timedelta(days=9, seconds=32148))
('0184d4ddb259e1a4cfc2871888cf97b8', 'delivered', datetime.datetime(2017, 9, 1, 20, 4, 28), datetime.datetime(2017, 9, 13, 22, 17, 15), datetime.datetime(2017, 9, 4, 14, 5, 50), datetime.datetime(2017, 9, 9, 15, 12, 44), datetime.timedelta(days=9, seconds=29485))
('1378f9601350615613cc8832d6789c5d', 'delivered', datetime.datetime(2017, 9, 1, 20, 28, 2), datetime.datetime(2017, 9, 13, 22, 3, 51), datetime.datetime(2017, 9, 4, 18, 7, 55), datetime.datetime(2017, 9, 13,

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            order_id,
            order_status,
            order_purchase_timestamp,
            order_delivered_carrier_date,
            order_delivered_customer_date,
            order_delivered_carrier_date - order_delivered_customer_date AS gap
        FROM staging.orders
        WHERE order_delivered_customer_date < order_delivered_carrier_date
        ORDER BY gap DESC
        LIMIT 10;
    """))

    for row in result:
        print(row)

('c1e2bf2b7dd3309f2f5356c6b63968fa', 'delivered', datetime.datetime(2017, 2, 10, 10, 19, 10), datetime.datetime(2017, 3, 2, 17, 34, 26), datetime.datetime(2017, 2, 14, 15, 15, 57), datetime.timedelta(days=16, seconds=8309))
('fa3e37584f4fdb1ded0e0de700dfcb4e', 'delivered', datetime.datetime(2017, 7, 30, 19, 32, 23), datetime.datetime(2017, 8, 9, 18, 18, 43), datetime.datetime(2017, 8, 1, 21, 13, 1), datetime.timedelta(days=7, seconds=75942))
('29941903985f944b0ffc49c479c1547d', 'delivered', datetime.datetime(2017, 5, 29, 16, 16, 50), datetime.datetime(2017, 6, 9, 15, 7, 29), datetime.datetime(2017, 6, 2, 11, 9, 16), datetime.timedelta(days=7, seconds=14293))
('dceb62e8fa94b46006c9554fed743df0', 'delivered', datetime.datetime(2017, 7, 20, 20, 58, 5), datetime.datetime(2017, 8, 1, 18, 23, 30), datetime.datetime(2017, 7, 26, 18, 9, 10), datetime.timedelta(days=6, seconds=860))
('76458889992169d3135b264dc13aec67', 'delivered', datetime.datetime(2016, 10, 7, 10, 5, 16), datetime.datetime(20

In [100]:
#Validate order statuses

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            order_status,
            COUNT(*) AS order_count
        FROM staging.orders
        GROUP BY order_status
        ORDER BY order_count DESC;
    """))

    for row in result:
        print(row)

('delivered', 96478)
('shipped', 1107)
('canceled', 625)
('unavailable', 609)
('invoiced', 314)
('processing', 301)
('created', 5)
('approved', 2)


In [101]:
#Check delivery-date consistency by status: test whether delivered orders actually have a customer delivery timestamp.

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) FILTER (
                WHERE order_status = 'delivered'
                  AND order_delivered_customer_date IS NULL
            ) AS delivered_missing_delivery_date,

            COUNT(*) FILTER (
                WHERE order_status <> 'delivered'
                  AND order_delivered_customer_date IS NOT NULL
            ) AS nondelivered_with_delivery_date

        FROM staging.orders;
    """))

    print(result.fetchone())



(8, 6)


In [102]:
#find which statuses make up those 6 cases

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            order_status,
            COUNT(*) AS order_count
        FROM staging.orders
        WHERE order_status <> 'delivered'
          AND order_delivered_customer_date IS NOT NULL
        GROUP BY order_status
        ORDER BY order_count DESC;
    """))

    for row in result:
        print(row)

('canceled', 6)


In [103]:
#Check orders with no order items

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS orders_without_items
        FROM staging.orders o
        LEFT JOIN staging.order_items oi
            ON o.order_id = oi.order_id
        WHERE oi.order_id IS NULL;
    """))

    print(result.scalar())

775


In [104]:
#Check statuses of those 775 orders

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            o.order_status,
            COUNT(*) AS order_count
        FROM staging.orders o
        LEFT JOIN staging.order_items oi
            ON o.order_id = oi.order_id
        WHERE oi.order_id IS NULL
        GROUP BY o.order_status
        ORDER BY order_count DESC;
    """))

    for row in result:
        print(row)

('unavailable', 603)
('canceled', 164)
('created', 5)
('invoiced', 2)
('shipped', 1)


In [105]:
#Inspect the shipped order

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            o.order_id,
            o.customer_id,
            o.order_status,
            o.order_purchase_timestamp,
            o.order_approved_at,
            o.order_delivered_carrier_date,
            o.order_delivered_customer_date
        FROM staging.orders o
        LEFT JOIN staging.order_items oi
            ON o.order_id = oi.order_id
        WHERE oi.order_id IS NULL
          AND o.order_status = 'shipped';
    """))

    print(result.fetchone())

('a68ce1686d536ca72bd2dadc4b8671e5', 'd7bed5fac093a4136216072abaf599d5', 'shipped', datetime.datetime(2016, 10, 5, 1, 47, 40), datetime.datetime(2016, 10, 7, 3, 11, 22), datetime.datetime(2016, 11, 7, 16, 37, 37), None)


In [106]:
#Check orders with no payment record

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS orders_without_payment
        FROM staging.orders o
        LEFT JOIN staging.payments p
            ON o.order_id = p.order_id
        WHERE p.order_id IS NULL;
    """))

    print(result.scalar())

1


In [107]:
#inspect that order:

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            o.order_id,
            o.customer_id,
            o.order_status,
            o.order_purchase_timestamp,
            o.order_approved_at,
            o.order_delivered_customer_date
        FROM staging.orders o
        LEFT JOIN staging.payments p
            ON o.order_id = p.order_id
        WHERE p.order_id IS NULL;
    """))

    print(result.fetchone())

('bfbd0f9bdef84302105ad712db648a6c', '86dc2ffce2dfff336de2f386a786e574', 'delivered', datetime.datetime(2016, 9, 15, 12, 16, 38), datetime.datetime(2016, 9, 15, 12, 16, 38), datetime.datetime(2016, 11, 9, 7, 47, 38))


In [108]:
#Check orders with no review

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS orders_without_review
        FROM staging.orders o
        LEFT JOIN staging.reviews r
            ON o.order_id = r.order_id
        WHERE r.order_id IS NULL;
    """))

    print(result.scalar())

768


In [109]:
#review coverage by order status

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            o.order_status,
            COUNT(*) AS orders_without_review
        FROM staging.orders o
        LEFT JOIN staging.reviews r
            ON o.order_id = r.order_id
        WHERE r.order_id IS NULL
        GROUP BY o.order_status
        ORDER BY orders_without_review DESC;
    """))

    for row in result:
        print(row)

('delivered', 646)
('shipped', 75)
('canceled', 20)
('unavailable', 14)
('processing', 6)
('invoiced', 5)
('created', 2)


In [110]:
#how many orders have multiple reviews

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS orders_with_multiple_reviews,
            MAX(review_count) AS max_reviews_per_order
        FROM (
            SELECT
                order_id,
                COUNT(*) AS review_count
            FROM staging.reviews
            GROUP BY order_id
            HAVING COUNT(*) > 1
        ) x;
    """))

    print(result.fetchone())

(547, 3)


In [111]:
#conflicting review scores

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS orders_with_conflicting_scores
        FROM (
            SELECT
                order_id
            FROM staging.reviews
            GROUP BY order_id
            HAVING COUNT(*) > 1
               AND COUNT(DISTINCT review_score) > 1
        ) x;
    """))

    print(result.scalar())

202


In [112]:
#reviews created before delivery

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS reviews_before_delivery
        FROM staging.reviews r
        JOIN staging.orders o
            ON r.order_id = o.order_id
        WHERE o.order_delivered_customer_date IS NOT NULL
          AND r.review_creation_date < o.order_delivered_customer_date::date;
    """))

    print(result.scalar())

5127


In [113]:
with engine.connect() as connection:
    result = connection.execute(text("""
        WITH latest_reviews AS (
            SELECT *
            FROM (
                SELECT
                    r.*,
                    ROW_NUMBER() OVER (
                        PARTITION BY order_id
                        ORDER BY review_answer_timestamp DESC
                    ) AS rn
                FROM staging.reviews r
            ) x
            WHERE rn = 1
        )

        SELECT COUNT(*) AS reviews_before_delivery
        FROM latest_reviews r
        JOIN staging.orders o
            ON r.order_id = o.order_id
        WHERE o.order_delivered_customer_date IS NOT NULL
          AND r.review_creation_date < o.order_delivered_customer_date::date;
    """))

    print(result.scalar())

4977


In [114]:
#how many latest reviews we have

with engine.connect() as connection:
    result = connection.execute(text("""
        WITH latest_reviews AS (
            SELECT *
            FROM (
                SELECT
                    r.*,
                    ROW_NUMBER() OVER (
                        PARTITION BY order_id
                        ORDER BY review_answer_timestamp DESC
                    ) AS rn
                FROM staging.reviews r
            ) x
            WHERE rn = 1
        )

        SELECT COUNT(*)
        FROM latest_reviews;
    """))

    print(result.scalar())

98673


In [115]:
#Check whether the “latest review” rule is unambiguous

with engine.connect() as connection:
    result = connection.execute(text("""
        WITH max_times AS (
            SELECT
                order_id,
                MAX(review_answer_timestamp) AS latest_timestamp
            FROM staging.reviews
            GROUP BY order_id
        )

        SELECT COUNT(*) AS orders_with_tied_latest_reviews
        FROM (
            SELECT
                r.order_id
            FROM staging.reviews r
            JOIN max_times m
                ON r.order_id = m.order_id
               AND r.review_answer_timestamp = m.latest_timestamp
            GROUP BY r.order_id
            HAVING COUNT(*) > 1
        ) x;
    """))

    print(result.scalar())

0


In [116]:
#Validate customer_unique_id

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS unique_customer_ids,
            COUNT(DISTINCT customer_unique_id) AS unique_real_customers,
            COUNT(*) FILTER (
                WHERE customer_unique_id IS NULL
            ) AS null_unique_customers
        FROM staging.customers;
    """))

    print(result.fetchone())

(99441, 99441, 96096, 0)


In [117]:
#Count repeat customers in SQL

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*) AS repeat_customers
        FROM (
            SELECT
                customer_unique_id
            FROM staging.customers
            GROUP BY customer_unique_id
            HAVING COUNT(*) > 1
        ) x;
    """))

    print(result.scalar())

2997


In [119]:
#Calculate repeat customer rate

with engine.connect() as connection:
    result = connection.execute(text("""
        WITH customer_orders AS (
            SELECT
                customer_unique_id,
                COUNT(*) AS order_count
            FROM staging.customers
            GROUP BY customer_unique_id
        )

        SELECT
            COUNT(*) AS total_customers,

            COUNT(*) FILTER (WHERE order_count > 1) AS repeat_customers,

            ROUND(100.0 * COUNT(*) FILTER (WHERE order_count > 1)/ COUNT(*),2) AS repeat_customer_rate

        FROM customer_orders;
    """))

    print(result.fetchone())

(96096, 2997, Decimal('3.12'))


In [120]:
#Check order-frequency distribution

with engine.connect() as connection:
    result = connection.execute(text("""
        WITH customer_orders AS (
            SELECT
                customer_unique_id,
                COUNT(*) AS order_count
            FROM staging.customers
            GROUP BY customer_unique_id
        )

        SELECT
            order_count,
            COUNT(*) AS customer_count
        FROM customer_orders
        GROUP BY order_count
        ORDER BY order_count;
    """))

    for row in result:
        print(row)

(1, 93099)
(2, 2745)
(3, 203)
(4, 30)
(5, 8)
(6, 6)
(7, 3)
(9, 1)
(17, 1)


In [121]:
#Validate the usable order date range

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            MIN(order_purchase_timestamp) AS first_order,
            MAX(order_purchase_timestamp) AS last_order
        FROM staging.orders;
    """))

    print(result.fetchone())

(datetime.datetime(2016, 9, 4, 21, 15, 19), datetime.datetime(2018, 10, 17, 17, 30, 18))


In [122]:
#Check monthly order counts

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            DATE_TRUNC('month', order_purchase_timestamp)::date AS order_month,
            COUNT(*) AS order_count
        FROM staging.orders
        GROUP BY order_month
        ORDER BY order_month;
    """))

    for row in result:
        print(row)

(datetime.date(2016, 9, 1), 4)
(datetime.date(2016, 10, 1), 324)
(datetime.date(2016, 12, 1), 1)
(datetime.date(2017, 1, 1), 800)
(datetime.date(2017, 2, 1), 1780)
(datetime.date(2017, 3, 1), 2682)
(datetime.date(2017, 4, 1), 2404)
(datetime.date(2017, 5, 1), 3700)
(datetime.date(2017, 6, 1), 3245)
(datetime.date(2017, 7, 1), 4026)
(datetime.date(2017, 8, 1), 4331)
(datetime.date(2017, 9, 1), 4285)
(datetime.date(2017, 10, 1), 4631)
(datetime.date(2017, 11, 1), 7544)
(datetime.date(2017, 12, 1), 5673)
(datetime.date(2018, 1, 1), 7269)
(datetime.date(2018, 2, 1), 6728)
(datetime.date(2018, 3, 1), 7211)
(datetime.date(2018, 4, 1), 6939)
(datetime.date(2018, 5, 1), 6873)
(datetime.date(2018, 6, 1), 6167)
(datetime.date(2018, 7, 1), 6292)
(datetime.date(2018, 8, 1), 6512)
(datetime.date(2018, 9, 1), 16)
(datetime.date(2018, 10, 1), 4)


In [124]:
#generate the missing months

with engine.connect() as connection:
    result = connection.execute(text("""
        WITH months AS (
            SELECT generate_series(DATE '2016-09-01',DATE '2018-10-01', INTERVAL '1 month')::date AS month),

        monthly_orders AS (
            SELECT
                DATE_TRUNC('month', order_purchase_timestamp)::date AS month,
                COUNT(*) AS order_count
            FROM staging.orders
            GROUP BY 1
        )

        SELECT
            m.month,
            COALESCE(o.order_count, 0) AS order_count
        FROM months m
        LEFT JOIN monthly_orders o
            ON m.month = o.month
        ORDER BY m.month;
    """))

    for row in result:
        print(row)

(datetime.date(2016, 9, 1), 4)
(datetime.date(2016, 10, 1), 324)
(datetime.date(2016, 11, 1), 0)
(datetime.date(2016, 12, 1), 1)
(datetime.date(2017, 1, 1), 800)
(datetime.date(2017, 2, 1), 1780)
(datetime.date(2017, 3, 1), 2682)
(datetime.date(2017, 4, 1), 2404)
(datetime.date(2017, 5, 1), 3700)
(datetime.date(2017, 6, 1), 3245)
(datetime.date(2017, 7, 1), 4026)
(datetime.date(2017, 8, 1), 4331)
(datetime.date(2017, 9, 1), 4285)
(datetime.date(2017, 10, 1), 4631)
(datetime.date(2017, 11, 1), 7544)
(datetime.date(2017, 12, 1), 5673)
(datetime.date(2018, 1, 1), 7269)
(datetime.date(2018, 2, 1), 6728)
(datetime.date(2018, 3, 1), 7211)
(datetime.date(2018, 4, 1), 6939)
(datetime.date(2018, 5, 1), 6873)
(datetime.date(2018, 6, 1), 6167)
(datetime.date(2018, 7, 1), 6292)
(datetime.date(2018, 8, 1), 6512)
(datetime.date(2018, 9, 1), 16)
(datetime.date(2018, 10, 1), 4)


In [125]:
#Count orders inside the reliable analysis window

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) FILTER (
                WHERE order_purchase_timestamp >= TIMESTAMP '2017-01-01'
                  AND order_purchase_timestamp <  TIMESTAMP '2018-09-01'
            ) AS reliable_window_orders,

            COUNT(*) FILTER (
                WHERE order_purchase_timestamp < TIMESTAMP '2017-01-01'
                   OR order_purchase_timestamp >= TIMESTAMP '2018-09-01'
            ) AS boundary_period_orders

        FROM staging.orders;
    """))

    print(result.fetchone())

(99092, 349)
